# Q1 — ASR Pipeline: Real JoshTalks Dataset + Whisper Fine-Tuning

**Dataset**: ~10-hour Hindi conversational audio from JoshTalks GCS  
**Model**: `openai/whisper-small`  
**Goal**: Preprocess → Fine-tune → Evaluate on FLEURS Hindi test → Error Analysis → Fix

## Data Schema
`user_id, recording_id, language, duration, rec_url_gcp, transcription_url_gcp, metadata_url_gcp`

Each `transcription_url_gcp` points to a JSON of segments:  
`[{"start": 0.11, "end": 14.42, "speaker_id": 245746, "text": "..."}]`

In [ ]:
# ── Google Colab setup ────────────────────────────────────────────────────
# !pip install -q torch torchaudio transformers datasets accelerate evaluate
# !pip install -q librosa soundfile jiwer editdistance tqdm requests
# !pip install -q openai-whisper


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm import tqdm

from data_loader import JoshTalksDatasetLoader, load_fleurs_hindi_test, fetch_transcription_segments
from preprocessing import WhisperInputProcessor, HindiTextNormalizer, AudioPreprocessor
from whisper_finetune import WhisperFinetuner
from evaluation import compute_wer, compute_cer, evaluate_samples, save_wer_results, WERResult, compare_wer_results
from error_analysis import ErrorAnalyzer
from number_normalizer import HindiNumberNormalizer

print('All imports OK ✓')
print(f'PyTorch: {torch.__version__} | GPU: {torch.cuda.is_available()}')

---
## 1(a). Dataset Preprocessing

**Steps performed:**
1. Download manifest CSV from Google Sheets (columns: user_id, recording_id, duration, rec_url_gcp, transcription_url_gcp)
2. For each recording, fetch `*_transcription.json` → parse into (start, end, speaker_id, text) segments
3. Download full WAV from GCS, slice into per-segment audio chunks
4. Filter: remove segments <0.5s or >30s (Whisper limit)
5. Audio: mono, 16kHz resampled, peak-normalised, silence-trimmed
6. Text: normalise Devanagari (strip punctuation, diacritics cleanup, Devanagari digits → ASCII)
7. Split RECORDINGS (not segments) into train/validation to avoid speaker leakage

In [ ]:
# ── Load and inspect manifest ─────────────────────────────────────────────
loader = JoshTalksDatasetLoader(
    cache_dir='../data/raw',
    max_segment_duration=30.0,
    min_segment_duration=0.5,
    max_recordings=None,   # Set to e.g. 5 for a quick smoke-test
)

manifest = loader.load_manifest()
print(f'Total recordings in manifest: {len(manifest)}')
print(f'Columns: {list(manifest.columns)}')
display(manifest.head(3))

In [ ]:
# ── Inspect a single transcription JSON ───────────────────────────────────
sample_row = manifest.iloc[0]
segments = fetch_transcription_segments(sample_row.transcription_url_gcp)
print(f'Recording: {sample_row.recording_id} | Duration: {sample_row.duration}s')
print(f'Segments found: {len(segments)}')
print('\nFirst 3 segments:')
for s in segments[:3]:
    print(f'  [{s["start"]:.2f}s – {s["end"]:.2f}s] {s["text"]}')

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────
all_durations = []
all_texts = []
for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='Scanning transcriptions'):
    segs = fetch_transcription_segments(row.transcription_url_gcp)
    for s in segs:
        d = s['end'] - s['start']
        if 0.5 <= d <= 30.0:
            all_durations.append(d)
            all_texts.append(s['text'])

total_hrs = sum(all_durations) / 3600
print(f'\n=== Dataset Stats ===')
print(f'Total usable segments: {len(all_durations)}')
print(f'Total audio hours:     {total_hrs:.2f}h')
print(f'Avg segment duration:  {np.mean(all_durations):.2f}s')
print(f'Median text length:    {np.median([len(t.split()) for t in all_texts]):.0f} words')

In [ ]:
# ── Apply text normalisation and inspect ──────────────────────────────────
normalizer = HindiTextNormalizer()

print('Text normalisation examples:')
for raw in all_texts[:5]:
    norm = normalizer.normalize(raw)
    print(f'  RAW : {raw}')
    print(f'  NORM: {norm}')
    print()

In [ ]:
# ── Build HuggingFace DatasetDict ─────────────────────────────────────────
# ⚠️  This downloads all audio (~10h total compressed WAV) on first run.
# Files are cached in data/raw/ so subsequent runs are fast.
dataset = loader.build_dataset(manifest=manifest, val_fraction=0.1, random_seed=42)
print('Dataset splits:', {k: len(v) for k, v in dataset.items()})
print('Columns:', dataset['train'].column_names)

---
## 1(b). Feature Extraction — Whisper Preprocessing

In [ ]:
MODEL_NAME = 'openai/whisper-small'

input_processor = WhisperInputProcessor(
    model_name=MODEL_NAME,
    language='hindi',
    task='transcribe',
)

# Map over dataset — produces input_features (log-mel) + labels (token ids)
columns_to_remove = [c for c in dataset['train'].column_names
                     if c not in ('input_features', 'labels')]

processed = dataset.map(
    input_processor,
    remove_columns=columns_to_remove,
    num_proc=1,
    desc='Extracting Whisper features',
)
processed.set_format('torch')

print('Processed dataset:', {k: len(v) for k, v in processed.items()})
sample = processed['train'][0]
print(f'input_features shape: {sample["input_features"].shape}')  # (80, 3000)
print(f'labels length: {len(sample["labels"])}')

---
## 1(b-c). Baseline WER — Pretrained Whisper-Small on FLEURS Hindi

We evaluate on **FLEURS Hindi test** (a standard, public benchmark) per assignment instructions.

| Model | Hindi WER |
|---|---|
| Whisper Small (Pretrained) | **0.83** (from provided WER table) |
| FT Whisper Small | To be filled after training |

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Load pretrained Whisper-small (no fine-tuning)
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base_processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='hindi', task='transcribe')
base_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_model = base_model.to(device)

audio_preprocessor = AudioPreprocessor()
text_normalizer = HindiTextNormalizer()

def transcribe_model(model, processor, audio_dict):
    """Transcribe one audio dict using the given model."""
    wav = audio_preprocessor.process_array(
        audio_dict['array'], audio_dict['sampling_rate']
    )
    inputs = processor(wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        ids = model.generate(
            inputs['input_features'],
            language='hindi',
            task='transcribe',
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

print('Model ready on:', device)

In [ ]:
# Load FLEURS Hindi test set
fleurs_test = load_fleurs_hindi_test(cache_dir='../data/raw')
print(f'FLEURS test samples: {len(fleurs_test)}')

# Evaluate baseline on FLEURS (use a subset for speed)
FLEURS_EVAL_N = 100   # Set to len(fleurs_test) for full evaluation
fleurs_subset = fleurs_test.select(range(min(FLEURS_EVAL_N, len(fleurs_test))))

baseline_refs, baseline_preds = [], []
for sample in tqdm(fleurs_subset, desc='Baseline eval (FLEURS)'):
    ref  = text_normalizer.normalize(sample['transcription'])
    pred = text_normalizer.normalize(transcribe_model(base_model, base_processor, sample['audio']))
    baseline_refs.append(ref)
    baseline_preds.append(pred)

baseline_wer = compute_wer(baseline_refs, baseline_preds)
print(f'\nBaseline WER on FLEURS Hindi: {baseline_wer:.4f} ({baseline_wer*100:.1f}%)')
print('(Published baseline from assignment: 0.83 → 83.0%)')

---
## 1(b). Fine-Tune on JoshTalks Dataset

In [ ]:
finetuner = WhisperFinetuner(
    model_name=MODEL_NAME,
    output_dir='../outputs/whisper-josht-finetuned',
    language='hindi',
    task='transcribe',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=1e-5,
    warmup_steps=100,
    eval_steps=200,
    save_steps=200,
    max_steps=-1,                    # full training
    fp16=torch.cuda.is_available(),
)

finetuner.setup(processed)
print('Trainer ready.')

In [ ]:
# ⚠️ Full training takes ~2–4h on a T4 GPU.
# To load a pre-saved checkpoint instead:
#   finetuner.load_checkpoint('../outputs/whisper-josht-finetuned')

finetuner.train()
print('Fine-tuning complete!')

---
## 1(c). WER Results Table — FLEURS Hindi Test

In [ ]:
# Load fine-tuned model
CHECKPOINT = '../outputs/whisper-josht-finetuned'
finetuner.load_checkpoint(CHECKPOINT)
finetuner.model.eval()
ft_model = finetuner.model.to(device)
ft_proc   = finetuner.processor

ft_refs, ft_preds = [], []
for sample in tqdm(fleurs_subset, desc='Fine-tuned eval (FLEURS)'):
    ref  = text_normalizer.normalize(sample['transcription'])
    pred = text_normalizer.normalize(transcribe_model(ft_model, ft_proc, sample['audio']))
    ft_refs.append(ref)
    ft_preds.append(pred)

ft_wer = compute_wer(ft_refs, ft_preds)

# ── Structured WER table ──────────────────────────────────────────────────
wer_table = pd.DataFrame([
    {'Model': 'Whisper Small (Pretrained)', 'Test Set': 'FLEURS Hindi', 'WER': f'{baseline_wer:.4f}', 'WER%': f'{baseline_wer*100:.1f}%'},
    {'Model': 'FT Whisper Small (JoshTalks)', 'Test Set': 'FLEURS Hindi', 'WER': f'{ft_wer:.4f}', 'WER%': f'{ft_wer*100:.1f}%'},
])
print('\n=== WER Results Table ===')
display(wer_table)

# Save
baseline_result = WERResult(
    split='test', num_samples=len(fleurs_subset),
    wer=baseline_wer, cer=0.0, description='Whisper Small (Pretrained) | FLEURS Hindi'
)
ft_result = WERResult(
    split='test', num_samples=len(fleurs_subset),
    wer=ft_wer, cer=0.0, description='FT Whisper Small (JoshTalks) | FLEURS Hindi'
)
save_wer_results([baseline_result, ft_result], '../outputs/wer_results.csv')
print('Saved to outputs/wer_results.csv')

---
## 1(d–e). Systematic Error Sampling + Taxonomy

**Sampling strategy**: Stratified by WER severity.
- Sort all test samples by per-sample WER descending
- Take every N-th to cover the full error spectrum (not just worst cases)
- This avoids cherry-picking and ensures we see mild, medium, and severe errors

In [ ]:
from evaluation import evaluate_samples
from error_analysis import ErrorAnalyzer

sample_results = evaluate_samples(ft_refs, ft_preds)

# Stratified sampling: sort by WER severity, then every-N
errors_sorted = sorted(
    [s for s in sample_results if s.sample_wer > 0],
    key=lambda x: x.sample_wer,
    reverse=True,
)

analyzer = ErrorAnalyzer(n_samples=25, sampling_strategy='systematic', random_seed=42)
annotated = analyzer.analyse(sample_results)

taxonomy = analyzer.taxonomy_summary(annotated)
print('=== Error Taxonomy (from data) ===')
for etype, count in taxonomy.items():
    print(f'  {etype}: {count}')

analyzer.save(annotated, '../outputs/error_samples.csv')

In [ ]:
# ── Show 5 concrete examples per top category ─────────────────────────────
from collections import defaultdict
by_type = defaultdict(list)
for e in annotated:
    for t in e.error_types:
        by_type[t].append(e)

top_3 = list(taxonomy.keys())[:3]
for etype in top_3:
    print(f'\n── Error Type: {etype.upper()} ─────────────────')
    for ex in by_type[etype][:5]:
        print(f'  REF : {ex.reference}')
        print(f'  PRED: {ex.prediction}')
        print(f'  WHY : {ex.reasoning[:120]}')
        print()

---
## 1(f–g). Proposed Fixes + Implementation

### Top 3 Error Types & Fixes

| # | Error Type | Fix |
|---|---|---|
| 1 | **number_error** | Post-process: normalise number words↔digits in both ref + pred before WER |
| 2 | **phonetic** | Data augmentation with SpecAugment + increase training steps |
| 3 | **english_mix** | Fine-tune with code-switched training data; tag [EN] words before WER |

**Implemented Fix**: Number Normalisation (Fix #1)

In [ ]:
num_normalizer = HindiNumberNormalizer()

# Apply fix to the FLEURS test predictions
fixed_refs  = [num_normalizer.normalize(r) for r in ft_refs]
fixed_preds = [num_normalizer.normalize(p) for p in ft_preds]

fixed_wer = compute_wer(fixed_refs, fixed_preds)

print('=== Before/After Fix (Number Normalisation) ===')
print(f'  WER before fix: {ft_wer:.4f} ({ft_wer*100:.1f}%)')
print(f'  WER after fix:  {fixed_wer:.4f} ({fixed_wer*100:.1f}%)')
print(f'  Absolute reduction: {(ft_wer - fixed_wer)*100:.2f}%')

# Show targeted examples on number-error samples
print('\nBefore / After on number-error examples:')
num_error_examples = [e for e in annotated if 'number_error' in e.error_types][:4]
for ex in num_error_examples:
    r_fix = num_normalizer.normalize(ex.reference)
    p_fix = num_normalizer.normalize(ex.prediction)
    from jiwer import wer as _wer
    before = round(_wer([ex.reference], [ex.prediction]), 4)
    after  = round(_wer([r_fix], [p_fix]), 4)
    print(f'  REF:  {ex.reference}')
    print(f'  PRED: {ex.prediction}')
    print(f'  WER before: {before:.2f}  →  after: {after:.2f}')
    print()

In [ ]:
# ── Final WER comparison chart ────────────────────────────────────────────
models = ['Pretrained\nWhisper-small', 'Fine-tuned\n(JoshTalks)', 'FT + Num\nNorm']
wers   = [baseline_wer * 100, ft_wer * 100, fixed_wer * 100]
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, wers, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, wers):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('WER on FLEURS Hindi Test (%)', fontsize=12)
ax.set_title('Q1 — WER Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(wers) * 1.25)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/q1_wer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

display(pd.read_csv('../outputs/error_samples.csv').head(10))